In [3]:
import os
import shutil
from datetime import datetime

import json
import matplotlib.pyplot as plt
import numpy as np

In [4]:
def backup_config(config_filename):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename_only = os.path.basename(config_filename)
    name, ext = os.path.splitext(filename_only)
    backup_name = f"{name}_{timestamp}{ext}"
    backup_path = os.path.join('config_backups', backup_name)
    shutil.copy2(config_filename, backup_path)

def update_config(config_filename, new_data, keys, save_backup=True):

    if save_backup:
        backup_config(config_filename)

    with open(config_filename, 'r') as f:
        config = json.load(f)
    
    data = config
    for i in range(len(keys)-1):
        key = keys[i]

        if key not in data:
            data[key] = {}
        data = data[key]

    data[keys[-1]] = new_data


    
    with open(config_filename, 'w') as f:
        json.dump(config, f, indent=4)

def import_config(config_filename_from, config_filename_to, keys_from, keys_to=None, save_backup=True):
    data_to_import = read_config(config_filename_from, keys_from)
    if keys_to is None:
        keys_to = keys_from
    update_config(config_filename_to, data_to_import, keys_to, save_backup=save_backup)

def read_config(config_filename, keys=None):
    with open(config_filename, 'r') as f:
        config = json.load(f)
    
    data = config
    if keys is not None:
        for i in range(len(keys)-1):
            key = keys[i]
            data = data[key]

        return data[keys[-1]]
    else:
        return data
        
def remove_field(config_filename, keys, save_backup=False):
    if save_backup:
        backup_config(config_filename)

    with open(config_filename, 'r') as f:
        config = json.load(f)
    
    data = config
    for i in range(len(keys)-1):
        key = keys[i]
        data = data[key]

    del data[keys[-1]]
    
    with open(config_filename, 'w') as f:
        json.dump(config, f, indent=4)


In [5]:
# ### testing

# update_config(config_filename, {1:2, 3:5}, ('coupling_point_1', 'data_config', 'test'))


In [6]:
config_filename = 'bond_order_config.json'

correlations_config_filename = '../current_correlations/current_correlations_config.json'


In [7]:
coupling_point = 'coupling_point_2_retake'
eigenstate = 'highest'

# 1. Experimental Parameters

In [9]:
### import coupling, T2, and simulation parameters from correlations config


import_config(correlations_config_filename, config_filename, (coupling_point, 'coupling'), save_backup=True)
import_config(correlations_config_filename, config_filename, (coupling_point, 'coupling_error'), save_backup=True)
import_config(correlations_config_filename, config_filename, (coupling_point, 'T2s'), save_backup=True)
import_config(correlations_config_filename, config_filename, (coupling_point, 'simulation_config'), save_backup=True)

KeyError: 'coupling_point_2_retake'

In [ ]:
### update measurement detunings

# get measurement detunings from the ((1,2),(3,4)) and ((2,3),(4,5)) correlations config specifically

correlation_config_data = read_config(correlations_config_filename, (coupling_point, 'data_config'))

print(correlation_config_data.keys())
print(correlation_config_data['((1, 2), (3, 4))'].keys())
print(correlation_config_data['((1, 2), (3, 4))']['measurement_detunings'])

bond_order_data = {}
bond_order_data['12-34-56-78'] = {}
bond_order_data['12-34-56-78']['measurement_detunings'] = correlation_config_data['((1, 2), (3, 4))']['measurement_detunings']

bond_order_data['1-23-45-67-8'] = {}
bond_order_data['1-23-45-67-8']['measurement_detunings'] = correlation_config_data['((2, 3), (4, 5))']['measurement_detunings']

update_config(config_filename, bond_order_data, (coupling_point, 'data_config'), save_backup=True)

dict_keys(['((1, 2), (3, 4))', '((1, 2), (5, 6))', '((1, 2), (7, 8))', '((3, 4), (5, 6))', '((3, 4), (7, 8))', '((5, 6), (7, 8))', '((2, 3), (4, 5))', '((2, 3), (6, 7))', '((4, 5), (6, 7))', '((1, 2), (4, 5))', '((2, 3), (5, 6))', '((3, 4), (6, 7))', '((4, 5), (7, 8))', '((1, 2), (6, 7))', '((2, 3), (7, 8))', 'post_select', 'confusion_matrix_correct'])
dict_keys(['configuration', 'beamsplitter_time', 'measurement_detunings', 'highest'])
[-1256.6370614359173, -1256.6370614359173, 3141.592653589793, 3141.592653589793, -628.3185307179587, -628.3185307179587, 1884.9555921538758, 1884.9555921538758]


In [ ]:
# #### ARCHIVE ######

# coupling point 1
# configuration_to_date_code_highest['12-34-56-78'] = ('2025', '12', '08', '10', '24', '56') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2025', '12', '09', '15', '33', '54') # high quality 1-23-45-67-8

# # coupling point 2

# configuration_to_date_code_highest['12-34-56-78'] = ('2025', '11', '11', '10', '35', '08') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2025', '11', '13', '13', '57', '01') # high quality 1-23-45-67-8

# # coupling point 2 retake 1

# configuration_to_date_code_highest['12-34-56-78'] = ('2026', '03', '01', '16', '34', '42') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '03', '01', '17', '04', '27') # high quality 1-23-45-67-8


# # coupling point 3
# configuration_to_date_code_highest['12-34-56-78'] = ('2026', '01', '11', '12', '48', '53') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '01', '13', '21', '39', '59') # high quality 1-23-45-67-8


# # coupling point 4

# configuration_to_date_code_highest['12-34-56-78'] = ('2025', '11', '18', '14', '03', '15') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2025', '11', '18', '22', '23', '42') # high quality 1-23-45-67-8

# # coupling point 5
# configuration_to_date_code_highest['12-34-56-78'] = ('2026', '01', '26', '22', '34', '54') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '01', '27', '14', '54', '46') # high quality 1-23-45-67-8


# # coupling point 6
# configuration_to_date_code_highest['12-34-56-78'] = ('2026', '02', '13', '18', '47', '42') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '02', '13', '19', '39', '55') # high quality 1-23-45-67-8

# # coupling point 6 retake 2/24/26
# configuration_to_date_code_highest['12-34-56-78'] = ('2026', '02', '24', '14', '48', '32') # high quality 12-34-56-78
# configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '02', '24', '15', '18', '48') # high quality 1-23-45-67-8


In [11]:
### add h5py filenames of measurement files



configuration_to_date_code_highest = {}
configuration_to_date_code_lowest = {}
configuration_to_date_code_disordered = {}

eigenstate_dict = {'highest': configuration_to_date_code_highest,
                   'lowest': configuration_to_date_code_lowest,
                   'disordered': configuration_to_date_code_disordered}



configuration_to_date_code_highest['12-34-56-78'] = ('2026', '03', '01', '16', '34', '42') # high quality 12-34-56-78
configuration_to_date_code_highest['1-23-45-67-8'] = ('2026', '03', '01', '17', '04', '27') # high quality 1-23-45-67-8

keys = (coupling_point, 'data_config')


backup_config(config_filename)
for eigenstate in eigenstate_dict:
    for configuration in eigenstate_dict[eigenstate]:

        date_code = eigenstate_dict[eigenstate][configuration]

        date_code_dict = {}
        date_code_dict['year'] = str(date_code[0])
        date_code_dict['month'] = str(date_code[1])
        date_code_dict['day'] = str(date_code[2])
        date_code_dict['hour'] = str(date_code[3])
        date_code_dict['minute'] = str(date_code[4])
        date_code_dict['second'] = str(date_code[5])



        update_config(config_filename, date_code_dict, keys + (str(configuration), eigenstate), save_backup=False)

In [12]:
### add filename base for each measurement

filename_base = None
# filename_base = 'RampBeamsplitterCorrelationsR'
# filename_base = 'RampBeamsplitterCleanTiming'
# filename_base = 'RampDoubleJumpCurrentCorrelations'
filename_base = 'BSClean_Correlations'

keys = (coupling_point, 'data_config')
eigenstate = 'highest'

correlation_pairs_to_update = []

configurations_to_update = ['12-34-56-78', '1-23-45-67-8']

backup_config(config_filename)
for configuration in configurations_to_update:
    update_config(config_filename, filename_base, keys + (configuration, eigenstate, 'filename_base'), save_backup=False)

In [13]:
### add parity for each qubit pair

# each qubit has a different parity depending on the relative ordering of the qubits 
# while we are idling to apply the z gate. This changes if they gain a phase of pi/2 or -pi/2

    
configuration_to_parity = {}
configuration_to_parity['12-34-56-78'] = [1,1,1,1]
configuration_to_parity['1-23-45-67-8'] = [1,1,1]

keys = (coupling_point, 'data_config')

backup_config(config_filename)
for configuration in configuration_to_parity:
        update_config(config_filename, configuration_to_parity[configuration], keys + (str(configuration), 'parity'), save_backup=False)



In [22]:
# ### set post selection and confusion matrix correction options

# post_select = True
# confusion_matrix_correct = True

# keys = (coupling_point, 'data_config')

# update_config(config_filename, post_select, keys + ('post_select',), save_backup=True)
# update_config(config_filename, confusion_matrix_correct, keys + ('confusion_matrix_correct',), save_backup=False)

# 2. Simulation Parameters

In [14]:

simulation_config_dict = {
    'num_levels': 5,
    'num_qubits': 8,
    'num_particles': 4,
    'U': -180,
    'psi0': -1,
    'T1': 40,
    'T2': 2,
    'time_start': 0,
    'time_stop': 0.16,
    'time_num_points': 201
}


keys = (coupling_point, 'simulation_config',)
update_config(config_filename, simulation_config_dict, keys, save_backup=True)



# 3. Edits

### 3.1 Delete

In [ ]:
# ### CAREFUL!! ### 
# # only run to remove certain sections of the json file

# backup_config(config_filename)
# labels = ['year', 'month', 'day', 'hour', 'minute', 'second']
# for correlation_pair in correlation_pair_to_configuration:
#     for label in labels:
#         keys = ('coupling_point_2', 'data_config', str(correlation_pair), label)

#         remove_field(config_filename, keys)

In [ ]:
# ### CAREFUL!! ### 
# # only run to remove certain sections of the json file

# backup_config(config_filename)
# for correlation_pair in correlation_pair_to_configuration:
#     keys = ('coupling_point_4', 'data_config', str(correlation_pair), 'disordered')

#     remove_field(config_filename, keys)

In [8]:
loaded_correlations_config = read_config('../current_correlations/current_correlations_config.json')

for coupling_point in loaded_correlations_config:
    coupling_dict = loaded_correlations_config[coupling_point]['coupling']
    print(coupling_dict)
   
    rung_couplings = []
    leg_couplings = []
    for i in range(1,8):
        rung_couplings.append(coupling_dict[f'J_{i}{i+1}'])
        if i < 7:
            leg_couplings.append(coupling_dict[f'J_{i}{i+2}'])
    

    average_rung_coupling = np.mean(rung_couplings)
    average_leg_coupling = np.mean(leg_couplings)
    print(f'Average rung coupling: {average_rung_coupling:.2f}')
    print(f'Average leg coupling: {average_leg_coupling:.2f}')

    coupling_dict['average_rung_coupling'] = average_rung_coupling
    coupling_dict['average_leg_coupling'] = average_leg_coupling

    update_config(config_filename, coupling_dict, (coupling_point, 'coupling'), save_backup=True)

{'J_13': -6.85, 'J_24': -6.76, 'J_35': -7.17, 'J_46': -7.3, 'J_57': -7.08, 'J_68': -6.82, 'J_12': -5.82, 'J_23': -5.87, 'J_34': -5.53, 'J_45': -5.79, 'J_56': -5.55, 'J_67': -5.83, 'J_78': -5.85, 'average_rung_coupling': -5.748571428571429, 'average_leg_coupling': -6.996666666666667}
Average rung coupling: -5.75
Average leg coupling: -7.00
{'J_13': -11.99, 'J_24': -12.72, 'J_35': -12.35, 'J_46': -11.27, 'J_57': -12.42, 'J_68': -12.54, 'J_12': -6.41, 'J_23': -6.15, 'J_34': -5.8, 'J_45': -5.95, 'J_56': -5.42, 'J_67': -6.26, 'J_78': -6.35, 'average_rung_coupling': -6.048571428571428, 'average_leg_coupling': -12.214999999999998}
Average rung coupling: -6.05
Average leg coupling: -12.21
{'J_13': -17.31, 'J_24': -17.49, 'J_35': -17.48, 'J_46': -17.49, 'J_57': -17.38, 'J_68': -17.33, 'J_12': -5.24, 'J_23': -4.99, 'J_34': -4.51, 'J_45': -4.71, 'J_56': -4.47, 'J_67': -5.05, 'J_78': -5.25, 'average_rung_coupling': -4.888571428571429, 'average_leg_coupling': -17.41333333333333}
Average rung coupli